# 31 - Fourier-space stacked tSZ profile: per-cluster $\tilde y(\ell)$ vs Arnaud A10

Notebook 03 stacked the *real-space* Compton-$y$ profile $y(\theta)$ of FLAMINGO
clusters and compared its shape against the projected Arnaud et al. (2010) GNFW
profile. Here we do the **Fourier-space** analogue.

**Strategy** (per cluster):

1. Locate the cluster centre $(\mathrm{ra}, \mathrm{dec})$ on the high-resolution
   $y$-map.
2. Cut a **square stamp of side $6\,R_{500c}$** centred on the cluster, i.e. the
   profile is captured out to a projected radius $\theta = 3\,\theta_{500}$.
   The stamp is a gnomonic (tangent-plane) projection of the HEALPix map.
3. Take the **2-D Fourier transform** of the stamp. For an azimuthally symmetric
   profile this is a **Hankel transform**, $\tilde y(\ell) = 2\pi\int \theta\,
   y(\theta)\,J_0(\ell\theta)\,d\theta$. Azimuthally averaging the 2-D FFT gives
   the per-cluster form factor $\tilde y(\ell)$.

Each stamp is **scaled to the cluster's own $\theta_{500}$** with a fixed number
of pixels $N$, so the dimensionless multipole grid $u \equiv \ell\,\theta_{500} =
\ell/\ell_{500}$ is **identical for every cluster**. Normalising each $\tilde
y(\ell)$ by its $\ell\to 0$ value (the DC mode $=$ total $Y$ in the stamp) removes
the per-cluster amplitude, leaving a pure shape that can be stacked directly.

**Theory.** By the projection-slice theorem the 2-D Hankel transform of the
projected A10 shape equals the pressure form factor used in `hmfast`. We compute
it directly from the repo's `projected_shape` (the A10 GNFW projected to Compton-y),
truncated at the same aperture $x<3$:
$$\frac{\tilde y(u)}{\tilde y(0)} = \frac{\int_0^3 x\,g(x)\,J_0(u x)\,dx}{\int_0^3 x\,g(x)\,dx},
\qquad x=\theta/\theta_{500},\; g=\texttt{projected\_shape}.$$


In [1]:
import os

# Keep JAX off most of the GPU; this notebook is mostly NumPy/pixell.
os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"] = "0.10"

import numpy as np
import matplotlib.pyplot as plt

# Publication-quality plot defaults
plt.rcParams.update({
    "text.usetex": False,
    "font.family": "serif",
    "font.size": 11,
    "axes.labelsize": 12,
    "axes.titlesize": 12,
    "legend.fontsize": 10,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "figure.dpi": 100,
    "savefig.dpi": 300,
    "axes.linewidth": 0.8,
    "xtick.direction": "in",
    "ytick.direction": "in",
    "xtick.top": True,
    "ytick.right": True,
})

from scipy.special import j0

from pixell import enmap, reproject

from flamingo import paths
from flamingo.maps import read_map
from flamingo.catalogue import load_catalogue
from flamingo.geometry import ARCMIN_PER_RAD
from flamingo.profiles import projected_shape

FIGDIR = "../figures/nb31_fourier_stacked_tsz_profile"
os.makedirs(FIGDIR, exist_ok=True)


In [2]:
# High-resolution (NSIDE=16384, ~0.21 arcmin pixels) unlensed y-map: needed so the
# scaled stamps are well resolved even for the largest clusters.
PATH_HIGHRES = "/rds/rds-lxu/tsz_project/flamingo_highres_maps/y_unlensed_L2p8_m9_lc0_nside16384.fits"

ymap = read_map(PATH_HIGHRES)
df = load_catalogue(paths.HYDRO_CATALOGUE)
print("map pixels", ymap.size, "| clusters", len(df))

map pixels 3221225472 | clusters 1555542


## Per-cluster Fourier transform

`cluster_yell` builds a tangent-plane geometry of side $6\,\theta_{500}$ with a
fixed $N\times N$ grid, samples the HEALPix $y$-map onto it (spline
interpolation, no coordinate rotation: the catalogue positions are already in the
map's native frame), FFTs the stamp, and azimuthally averages $|\tilde y(\ell)|$
into common bins of $u=\ell\,\theta_{500}$. Normalising by the DC mode gives the
dimensionless shape.

In [3]:
N_PIX = 128          # stamp is N_PIX x N_PIX
X_HALF = 3.0         # half-width in units of theta_500 -> square side = 6 R_500c
U_EDGES = np.linspace(0.0, 25.0, 41)        # bins in u = ell * theta_500
U_MID = 0.5 * (U_EDGES[1:] + U_EDGES[:-1])


def cluster_stamp(theta500_arcmin, lon_deg, lat_deg):
    '''Gnomonic y stamp of side 6 theta_500 (N_PIX x N_PIX) centred on a cluster.'''
    theta500 = theta500_arcmin / ARCMIN_PER_RAD
    res = 2.0 * X_HALF * theta500 / N_PIX
    dec, ra = np.deg2rad(lat_deg), np.deg2rad(lon_deg)
    shape, wcs = enmap.geometry(pos=(dec, ra), shape=(N_PIX, N_PIX), res=res, proj="tan")
    return reproject.healpix2map(ymap, shape, wcs, method="spline", order=1, rot=None)


def cluster_yell(theta500_arcmin, lon_deg, lat_deg):
    '''Per-cluster normalised Fourier form factor |y(ell)|/|y(0)| on the U grid.

    Returns nan in empty u-bins.
    '''
    theta500 = theta500_arcmin / ARCMIN_PER_RAD
    stamp = cluster_stamp(theta500_arcmin, lon_deg, lat_deg)

    fft = enmap.fft(stamp, normalize=False)
    amp = np.abs(np.asarray(fft))
    amp /= amp[0, 0]                          # normalise by DC mode (total Y)

    ly, lx = stamp.lmap()
    u = (np.hypot(np.asarray(ly), np.asarray(lx)) * theta500).ravel()
    amp = amp.ravel()

    prof = np.full(U_MID.size, np.nan)
    idx = np.digitize(u, U_EDGES) - 1
    for b in range(U_MID.size):
        sel = idx == b
        if sel.any():
            prof[b] = amp[sel].mean()
    return prof

## Sample selection

We use low-redshift ($z<0.3$), most-massive clusters so that $\theta_{500}$ is
large (several arcmin) and each $6\,\theta_{500}$ stamp spans many native map
pixels, giving a clean high-$\ell$ measurement.

In [4]:
N_STACK = 400
sample = df[df["z"] < 0.3].nlargest(N_STACK, "M_500c_Msun")
print(f"N={len(sample)}  "
      f"M500=[{sample['M_500c_Msun'].min():.2e}, {sample['M_500c_Msun'].max():.2e}]  "
      f"theta500=[{sample['theta500_arcmin'].min():.2f}, {sample['theta500_arcmin'].max():.2f}] arcmin")

N=400  M500=[6.23e+14, 2.07e+15]  theta500=[4.39, 44.74] arcmin


In [5]:
rows = []
for _, row in sample.iterrows():
    rows.append(cluster_yell(row["theta500_arcmin"], row["lon_rot_deg"], row["lat_rot_deg"]))
G = np.vstack(rows)                          # [n_clusters, n_u]

n = np.sum(np.isfinite(G), axis=0)
fhat = np.nanmean(G, axis=0)
sem = np.nanstd(G, axis=0) / np.sqrt(np.maximum(n, 1))
p16, p84 = np.nanpercentile(G, [16, 84], axis=0)
print("stacked", G.shape[0], "clusters")

stacked 400 clusters


## A10 theory form factor

Hankel transform of the projected A10 shape, truncated at $x<3$ to match the
stamp aperture. We also show the un-truncated ($x<5$) transform to expose the
effect of the hard aperture at low $\ell$.

In [6]:
def a10_form_factor(u_grid, x_max=3.0, n_x=4000):
    '''Normalised Hankel transform of the projected A10 shape, aperture x < x_max.'''
    x = np.linspace(1e-3, x_max, n_x)
    g = np.asarray(projected_shape(x))
    norm = np.trapezoid(x * g, x)
    return np.array([np.trapezoid(x * g * j0(u * x), x) for u in u_grid]) / norm


theory_x3 = a10_form_factor(U_MID, x_max=3.0)
theory_x5 = a10_form_factor(U_MID, x_max=5.0)

An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.


In [7]:
fig, ax = plt.subplots(figsize=(7.2, 5.4))
ax.fill_between(U_MID, p16, p84, color="crimson", alpha=0.15,
                label="16-84 percentile (cluster scatter)")
ax.fill_between(U_MID, fhat - sem, fhat + sem, color="crimson", alpha=0.5,
                label="SEM (error on stack)")
ax.plot(U_MID, fhat, "o", ms=4, color="crimson",
        label=f"map per-cluster FFT, stacked (N={G.shape[0]})")
ax.plot(U_MID, theory_x3, "k-", lw=1.8, label=r"A10 Hankel ($x<3$, matched aperture)")
ax.plot(U_MID, theory_x5, "k:", lw=1.4, label=r"A10 Hankel ($x<5$)")
ax.set_xlabel(r"$u=\ell\,\theta_{500}=\ell/\ell_{500}$")
ax.set_ylabel(r"$\tilde y(\ell)/\tilde y(0)$")
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_ylim(2e-3, 1.5)
ax.set_xlim(0.01, 25)
ax.set_title("Fourier-space stacked tSZ profile vs Arnaud A10")
ax.legend(fontsize=8.5)
ax.grid(alpha=0.3, which='both')
fig.tight_layout()
fig.savefig(f"{FIGDIR}/fourier_stack_vs_a10.pdf")
fig.savefig(f"{FIGDIR}/fourier_stack_vs_a10.png"), dpi=300)
plt.show()


## Residual ratio

Ratio of the stacked map form factor to the matched-aperture A10 theory.

In [8]:
ratio = fhat / theory_x3
ratio_err = sem / theory_x3

fig, ax = plt.subplots(figsize=(7.2, 4.2))
ax.axhline(1.0, color="grey", ls=":", lw=1)
ax.errorbar(U_MID, ratio, yerr=ratio_err, fmt="o", ms=4, color="crimson", capsize=2)
ax.set_xlabel(r"$u=\ell\,\theta_{500}$")
ax.set_ylabel(r"map / A10 ($x<3$)")
ax.set_xlim(0, 25)
ax.set_ylim(0.5, 1.6)
ax.set_title("Fourier form-factor residual")
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(f"{FIGDIR}/fourier_residual.pdf")
fig.savefig(f"{FIGDIR}/fourier_residual.png"), dpi=300)
plt.show()


## Example stamp and its 2-D Fourier transform

Visual check that the stamp is centred and the FFT is the expected azimuthally
symmetric Hankel kernel.

In [9]:
ex = sample.iloc[0]
theta500 = ex["theta500_arcmin"] / ARCMIN_PER_RAD
half = X_HALF * theta500
res = 2.0 * half / N_PIX
shape, wcs = enmap.geometry(pos=(np.deg2rad(ex["lat_rot_deg"]), np.deg2rad(ex["lon_rot_deg"])),
                            shape=(N_PIX, N_PIX), res=res, proj="tan")
stamp = reproject.healpix2map(ymap, shape, wcs, method="spline", order=1, rot=None)
fft = np.fft.fftshift(np.abs(np.asarray(enmap.fft(stamp, normalize=False))))

fig, axes = plt.subplots(1, 2, figsize=(11, 4.6))
ext = [-X_HALF, X_HALF, -X_HALF, X_HALF]
im0 = axes[0].imshow(np.asarray(stamp), origin="lower", extent=ext, cmap="viridis")
axes[0].set_title(f"y stamp (M500={ex['M_500c_Msun']:.1e}, z={ex['z']:.2f})")
axes[0].set_xlabel(r"$\theta_x/\theta_{500}$"); axes[0].set_ylabel(r"$\theta_y/\theta_{500}$")
fig.colorbar(im0, ax=axes[0], fraction=0.046)
im1 = axes[1].imshow(np.log10(fft + 1e-12), origin="lower", cmap="viridis")
axes[1].set_title(r"$\log_{10}|\tilde y(\ell)|$ (fftshift)")
axes[1].set_xlabel(r"$\ell_x$ pixel"); axes[1].set_ylabel(r"$\ell_y$ pixel")
fig.colorbar(im1, ax=axes[1], fraction=0.046)
fig.tight_layout()
fig.savefig(f"{FIGDIR}/example_stamp_fft.png"), dpi=300)
plt.show()


## Cross-check: pixell's native annular binning (`enmap.lbin`)

The stack above used a hand-rolled annular average (`np.digitize`) in the
cluster-scaled multipole $u=\ell\,\theta_{500}$, because the shared dimensionless
grid is the whole point. pixell ships the idiomatic radial binner `enmap.lbin`,
which averages a 2-D $\ell$-space map into annuli of raw multipole (and
`enmap.calc_ps2d` builds the 2-D power $|\tilde y(\ell)|^2$). Here we bin the
example cluster with `enmap.lbin`, convert its bin centres
$\ell \to u=\ell\,\theta_{500}$, and confirm it lands on top of the hand-binned
curve, i.e. the manual binning reproduces the pixell technique.

In [10]:
theta500 = ex["theta500_arcmin"] / ARCMIN_PER_RAD
amp2d = enmap.enmap(np.abs(np.asarray(enmap.fft(stamp, normalize=False))), stamp.wcs)
amp2d = amp2d / amp2d[0, 0]                          # normalise by DC mode

dl = 2.0 * np.pi / (2.0 * X_HALF * theta500)         # fundamental mode spacing
prof_lbin, bins = enmap.lbin(amp2d, bsize=2.0 * dl, return_bins=True)
u_lbin = bins[0] * theta500                          # bins[0] = bin centres in raw ell

# hand-binned profile for the same cluster, on the U grid
prof_manual = cluster_yell(ex["theta500_arcmin"], ex["lon_rot_deg"], ex["lat_rot_deg"])

fig, ax = plt.subplots(figsize=(7.0, 5.0))
ax.plot(u_lbin, prof_lbin, "s", ms=6, mfc="none", color="navy", label="pixell enmap.lbin")
ax.plot(U_MID, prof_manual, "o", ms=3, color="crimson", label="hand-binned (np.digitize)")
ax.plot(U_MID, theory_x3, "k-", lw=1.4, label=r"A10 Hankel ($x<3$)")
ax.set_xlabel(r"$u=\ell\,\theta_{500}$")
ax.set_ylabel(r"$\tilde y(\ell)/\tilde y(0)$")
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_ylim(2e-3, 1.5)
ax.set_xlim(1e-1, 25)
ax.set_title(f"Single cluster: pixell lbin vs hand-binned (M500={ex['M_500c_Msun']:.1e})")
ax.legend(fontsize=9)
ax.grid(which="both", alpha=0.3)
fig.tight_layout()
fig.savefig(f"{FIGDIR}/lbin_crosscheck.png"), dpi=300)
plt.show()


## Gallery: several clusters and their Fourier profiles

A set of individual clusters spanning the mass range of the sample. The top rows
show each $y$ stamp (scaled to its own $\theta_{500}$); the bottom panel overlays
their individual Fourier form factors $\tilde y(\ell)/\tilde y(0)$ against the A10
Hankel curve. Per-cluster curves are noisier than the stack but each tracks A10
over the well-measured range.

In [11]:
N_GAL = 8
# Span the sample in mass: evenly spaced in rank from most to least massive.
gal_idx = np.linspace(0, len(sample) - 1, N_GAL).astype(int)
gallery = sample.iloc[gal_idx]

ncol = 4
nrow = int(np.ceil(N_GAL / ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=(3.0 * ncol, 3.0 * nrow))
ext = [-X_HALF, X_HALF, -X_HALF, X_HALF]
for ax, (_, row) in zip(axes.ravel(), gallery.iterrows()):
    st = np.asarray(cluster_stamp(row["theta500_arcmin"], row["lon_rot_deg"], row["lat_rot_deg"]))
    ax.imshow(st, origin="lower", extent=ext, cmap="viridis")
    ax.set_title(f"M500={row['M_500c_Msun']:.1e}\n"
                 rf"$z$={row['z']:.2f}, $\theta_{{500}}$={row['theta500_arcmin']:.1f}'",
                 fontsize=8)
    ax.set_xticks([-2, 0, 2]); ax.set_yticks([-2, 0, 2])
    ax.tick_params(labelsize=7)
for ax in axes.ravel()[N_GAL:]:
    ax.axis("off")
fig.supxlabel(r"$\theta_x/\theta_{500}$", fontsize=10)
fig.supylabel(r"$\theta_y/\theta_{500}$", fontsize=10)
fig.suptitle("Gallery of y stamps (side $6R_{500c}$)", fontsize=12)
fig.tight_layout()
fig.savefig(f"{FIGDIR}/gallery_stamps.png"), dpi=300)
plt.show()


In [12]:
fig, ax = plt.subplots(figsize=(7.6, 5.4))
colors = plt.cm.viridis(np.linspace(0, 0.9, N_GAL))
for c, (_, row) in zip(colors, gallery.iterrows()):
    prof = cluster_yell(row["theta500_arcmin"], row["lon_rot_deg"], row["lat_rot_deg"])
    ax.plot(U_MID, prof, "-o", ms=3, lw=1.0, color=c,
            label=f"M500={row['M_500c_Msun']:.1e}, z={row['z']:.2f}")
ax.plot(U_MID, theory_x3, "k-", lw=2.2, label="A10 Hankel ($x<3$)")
ax.set_xlabel(r"$u=\ell\,\theta_{500}=\ell/\ell_{500}$")
ax.set_ylabel(r"$\tilde y(\ell)/\tilde y(0)$")
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlim(1e0, 25)
ax.set_ylim(2e-3, 1.5)
ax.set_title("Per-cluster Fourier form factors (log-log scale)")
ax.legend(fontsize=7.5, ncol=2)
ax.grid(alpha=0.3, which='both')
fig.tight_layout()
fig.savefig(f"{FIGDIR}/gallery_yell.png"), dpi=300)
fig.savefig(f"{FIGDIR}/gallery_yell.pdf")
plt.show()


## Notes

- The dimensionless grid $u=\ell\,\theta_{500}$ is shared across clusters because
  each stamp is scaled to its own $\theta_{500}$ at fixed $N_{\rm pix}$; the stack
  is therefore a simple per-bin mean.
- **Low-$\ell$ ($u\lesssim 4$):** the map form factor sits slightly below the A10
  curve. The square aperture collects extra flux in its corners (out to
  $x\simeq 3\sqrt2$) plus real 2-halo / neighbour signal, inflating the DC mode
  and suppressing the normalised low-$\ell$ amplitude.
- **High-$\ell$ ($u\gtrsim 18$):** $|\tilde y(\ell)|$ flattens to a small floor;
  taking the modulus rectifies pixel-window and interpolation noise. The clean
  comparison is $u\lesssim 15$, which already covers the bulk of the form factor.
- The matched-aperture A10 Hankel ($x<3$) is the correct theory comparison; the
  $x<5$ curve shows that the aperture mostly affects $u\lesssim 3$.


## Population at fixed mass and redshift: raw (un-rescaled) $\tilde y(\ell)$

The stack above scaled every cluster to its own $\theta_{500}$ and divided out the
amplitude to expose a single universal shape. Here we do the opposite: we fix
**both** the redshift (a narrow slice $z\in[0.04,0.05]$, the closest to $z=0$ with
enough clusters) **and** the mass (a narrow $M_{500c}$ bin), so $\theta_{500}$ is
constant to a few percent across the population. We then plot the **raw**
$\tilde y(\ell)$ in absolute units ($\mathrm{sr}$) against the true multipole
$\ell$, with **no $\theta_{500}$ scaling and no DC normalisation**. Every cluster
is transformed on one common angular grid, so the overlaid curves show the genuine
population of clusters at the same $M$ and $z$, including the cluster-to-cluster
amplitude scatter the rescaled stack removes by construction. The black curve is
the A10 Hankel form factor mapped to absolute units by scaling to the population's
mean $\tilde y(0)$ (total $Y$) at the median $\theta_{500}$.

In [13]:
# --- Fixed (M, z) population: raw, un-rescaled Fourier profiles ---
# Pin BOTH the redshift (a narrow z slice near z=0) and the mass (a narrow M_500c
# bin) so theta_500 is nearly constant (a few percent) across the population. A
# whole light-cone shell is too wide at z ~ 0: angular size changes steeply with
# distance there, so we slice z directly instead. Unlike the stack above we do NOT
# scale each stamp to its own theta_500 and do NOT normalise by the DC mode: every
# cluster shares one fixed angular grid, so the raw |y(ell)| curves overlay
# directly and show the genuine scatter of clusters at the same mass and redshift.

Z_LO, Z_HI = 0.04, 0.05                           # narrow z slice (closest to z=0)
M_LO, M_HI = 1.0e14, 1.3e14                        # narrow M_500c bin [Msun]

pop = df[(df["z"] >= Z_LO) & (df["z"] < Z_HI) &
         (df["M_500c_Msun"] >= M_LO) & (df["M_500c_Msun"] < M_HI)]
t500_scatter = 100.0 * pop["theta500_arcmin"].std() / pop["theta500_arcmin"].median()
print(f"N={len(pop)}  z=[{pop['z'].min():.3f},{pop['z'].max():.3f}]  "
      f"M500=[{pop['M_500c_Msun'].min():.2e},{pop['M_500c_Msun'].max():.2e}]  "
      f"theta500 median={pop['theta500_arcmin'].median():.2f} arcmin "
      f"(scatter {t500_scatter:.0f}%)")

# One fixed angular grid for the whole population (side = 6 x median theta_500),
# so the multipole grid is identical for every cluster -> raw curves are directly
# comparable without any rescaling.
HALF_ARCMIN = 3.0 * pop["theta500_arcmin"].median()
RES_FIX = (2.0 * HALF_ARCMIN / N_PIX) / ARCMIN_PER_RAD
PIXAREA = RES_FIX ** 2                            # solid angle per pixel [sr]
DL_FIX = 2.0 * np.pi / (2.0 * HALF_ARCMIN / ARCMIN_PER_RAD)   # fundamental mode spacing


def cluster_yell_raw(lon_deg, lat_deg):
    '''Raw azimuthally-averaged |y(ell)| [sr] on the shared fixed ell grid.

    No theta_500 scaling and no DC normalisation: the DC bin equals the
    integrated Compton-Y in the stamp, int y dOmega.
    '''
    dec, ra = np.deg2rad(lat_deg), np.deg2rad(lon_deg)
    shape, wcs = enmap.geometry(pos=(dec, ra), shape=(N_PIX, N_PIX), res=RES_FIX, proj="tan")
    stamp = reproject.healpix2map(ymap, shape, wcs, method="spline", order=1, rot=None)
    amp2d = enmap.enmap(np.abs(np.asarray(enmap.fft(stamp, normalize=False))) * PIXAREA, stamp.wcs)
    prof, bins = enmap.lbin(amp2d, bsize=2.0 * DL_FIX, return_bins=True)
    return bins[0], prof


ell_raw = None
rows_raw = []
for _, row in pop.iterrows():
    ell_raw, prof = cluster_yell_raw(row["lon_rot_deg"], row["lat_rot_deg"])
    rows_raw.append(prof)
Y_raw = np.vstack(rows_raw)                       # [n_clusters, n_ell], units of sr
print("raw population", Y_raw.shape, "| DC = Y [sr] median", f"{np.median(Y_raw[:, 0]):.3e}")

N=47  z=[0.041,0.050]  M500=[1.01e+14,1.30e+14]  theta500 median=13.05 arcmin (scatter 6%)
raw population (47, 45) | DC = Y [sr] median 2.090e-10


In [14]:
mean_raw = np.nanmean(Y_raw, axis=0)
p16r, p84r = np.nanpercentile(Y_raw, [16, 84], axis=0)

# A10 Hankel reference for this single (M, z): the normalised form factor mapped to
# absolute units by scaling to the population's mean Y(0) (DC mode) and converting
# u = ell * theta_500 with the population median theta_500.
theta500_med_rad = pop["theta500_arcmin"].median() / ARCMIN_PER_RAD
u_ref = ell_raw * theta500_med_rad
y_ref = mean_raw[0] * a10_form_factor(u_ref, x_max=3.0)

fig, ax = plt.subplots(figsize=(7.4, 5.4))
# individual clusters: the population at fixed (M, z), nothing rescaled
for prof in Y_raw:
    ax.plot(ell_raw, prof, "-", lw=0.5, color="grey", alpha=0.35)
ax.plot([], [], "-", lw=0.6, color="grey", alpha=0.6,
        label=f"individual clusters (N={Y_raw.shape[0]})")
ax.fill_between(ell_raw, p16r, p84r, color="crimson", alpha=0.2, label="16-84 percentile")
ax.plot(ell_raw, mean_raw, "o-", ms=4, color="crimson", label="population mean")
ax.plot(ell_raw, y_ref, "k-", lw=1.8,
        label=r"A10 Hankel (scaled to mean $Y$, $\theta_{500}$)")
ax.set_xlabel(r"$\ell$")
ax.set_ylabel(r"$\tilde y(\ell)\ \ [\mathrm{sr}]$")
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlim(ell_raw[0] * 0.8, 2e4)
ax.set_title(r"Raw $\tilde y(\ell)$ population at fixed $M$ and $z$ (no rescaling)"
             "\n"
             rf"$z\in[{Z_LO},{Z_HI}]$ ($\bar z={pop['z'].median():.3f}$), "
             rf"$M_{{500c}}\in[{M_LO:.1e},{M_HI:.1e}]\,M_\odot$")
ax.legend(fontsize=8.5)
ax.grid(alpha=0.3, which="both")
fig.tight_layout()
fig.savefig(f"{FIGDIR}/raw_yell_fixedMz_population.pdf")
fig.savefig(f"{FIGDIR}/raw_yell_fixedMz_population.png"), dpi=300)
plt.show()


## Raw $\tilde y(\ell)$ vs $\ell$ for different mass populations, at fixed redshift

Holding the redshift fixed (same narrow slice $z\in[0.04,0.05]$) we now vary the
mass. Each solid curve is the **raw**, un-rescaled $\tilde y(\ell)$ in absolute
units ($\mathrm{sr}$) stacked over a narrow $M_{500c}$ bin and plotted against the
true multipole $\ell$; the matching dashed curve is the A10 Hankel reference for
that mass, scaled to the bin's mean $\tilde y(0)$ (total $Y$) and median
$\theta_{500}$. Because nothing is rescaled, heavier clusters sit higher (larger
total $Y$) and turn over at lower $\ell$ (larger $\theta_{500}$), so the curves
separate in both amplitude and angular scale.

In [15]:
# --- Raw y(ell) vs ell for several mass populations at fixed z, with A10 refs ---
# Keep z constant (same narrow slice) and split it into M_500c bins. For each bin
# we stack the RAW |y(ell)| [sr] (no theta_500 scaling, no DC normalisation) on a
# per-bin angular grid sized to that bin's median theta_500, then overlay the A10
# Hankel reference scaled to the bin's mean Y(0) and median theta_500. Heavier
# clusters have larger total Y (higher curves) and larger theta_500 (turnover at
# lower ell), so the raw curves separate in both amplitude and scale.

MASS_BINS = [(0.7e14, 1.0e14), (1.0e14, 1.5e14), (1.5e14, 2.5e14), (2.5e14, 5.0e14)]
zslice = df[(df["z"] >= Z_LO) & (df["z"] < Z_HI)]


def raw_stack(sample, half_arcmin):
    '''Mean raw |y(ell)| [sr] and its ell grid for a sample, fixed angular grid.'''
    res = (2.0 * half_arcmin / N_PIX) / ARCMIN_PER_RAD
    pixarea = res ** 2
    dl = 2.0 * np.pi / (2.0 * half_arcmin / ARCMIN_PER_RAD)
    ell = None
    rows = []
    for _, r in sample.iterrows():
        dec, ra = np.deg2rad(r["lat_rot_deg"]), np.deg2rad(r["lon_rot_deg"])
        shp, wcs = enmap.geometry(pos=(dec, ra), shape=(N_PIX, N_PIX), res=res, proj="tan")
        stamp = reproject.healpix2map(ymap, shp, wcs, method="spline", order=1, rot=None)
        amp2d = enmap.enmap(np.abs(np.asarray(enmap.fft(stamp, normalize=False))) * pixarea, stamp.wcs)
        prof, bins = enmap.lbin(amp2d, bsize=2.0 * dl, return_bins=True)
        ell = bins[0]
        rows.append(prof)
    return ell, np.nanmean(np.vstack(rows), axis=0)


fig, ax = plt.subplots(figsize=(7.6, 5.4))
colors = plt.cm.plasma(np.linspace(0.1, 0.85, len(MASS_BINS)))
for c, (lo, hi) in zip(colors, MASS_BINS):
    mb = zslice[(zslice["M_500c_Msun"] >= lo) & (zslice["M_500c_Msun"] < hi)]
    if len(mb) > 60:
        mb = mb.sample(60, random_state=0)        # cap cost; the mean converges fast
    t500_med = mb["theta500_arcmin"].median()
    ell, ybar = raw_stack(mb, 3.0 * t500_med)
    # A10 Hankel reference for this M: scaled to the measured mean Y(0) and theta_500
    y_ref = ybar[0] * a10_form_factor(ell * (t500_med / ARCMIN_PER_RAD), x_max=3.0)
    ax.plot(ell, ybar, "-o", ms=3, lw=1.2, color=c,
            label=rf"$M_{{500c}}\in[{lo:.1e},{hi:.1e}]$ (N={len(mb)})")
    ax.plot(ell, y_ref, "--", lw=1.4, color=c)
ax.plot([], [], "k--", lw=1.4, label=r"A10 Hankel (scaled per $M$)")
ax.set_xlabel(r"$\ell$")
ax.set_ylabel(r"$\tilde y(\ell)\ \ [\mathrm{sr}]$")
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlim(1e2, 2e4)
ax.set_title(rf"Raw $\tilde y(\ell)$ vs mass at fixed $z$ "
             rf"($z\in[{Z_LO},{Z_HI}]$, $\bar z={zslice['z'].median():.3f}$)")
ax.legend(fontsize=8)
ax.grid(alpha=0.3, which="both")
fig.tight_layout()
fig.savefig(f"{FIGDIR}/raw_yell_vs_mass_fixedz.pdf")
fig.savefig(f"{FIGDIR}/raw_yell_vs_mass_fixedz.png"), dpi=300)
plt.show()
